# Lab 12: Statistical error estimation

Yesterday we generated a sequence of Ising configurations. Consecutive configurations share most of their spins, so consecutive measurements share information too.

How much information do we actually have? We'll measure the correlation, group measurements into blocks, and use jackknife and bootstrap resampling to put an uncertainty on a result.

Run the cells in order. This notebook only needs NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 12, "figure.figsize": (9, 3.2),
                     "figure.constrained_layout.use": True})

## Spread of the data, uncertainty of the mean

The standard deviation $s$ describes the spread of **individual measurements**. The standard error describes how much an **estimate** would vary if we repeated the whole experiment.

For $N$ independent measurements,

$$\bar x=\frac{1}{N}\sum_t x_t, \qquad
\operatorname{SE}_{\mathrm{iid}}(\bar x)=\frac{s}{\sqrt N}.$$

Here “iid” means independent and identically distributed. For correlated measurements, dividing by $\sqrt N$ can give an error bar that is much too small.

## A sequence whose answer we know

We'll use a small model of correlated measurements:

$$x_t=r x_{t-1}+\sqrt{1-r^2}\,\epsilon_t,
\qquad \epsilon_t\sim\mathcal N(0,1),\quad |r|<1.$$

The parameter $r$ controls how much of the previous value survives. With $x_0\sim\mathcal N(0,1)$, every $x_t$ has mean **0** and variance **1**. Changing $r$ changes the dependence between measurements, not their individual distribution.

This is a Gaussian Markov chain, often called AR(1). We know its stationary distribution, so we can start in it. An Ising simulation generally needs equilibration first; this example deliberately separates that issue from statistical error.

In [ ]:
def stationary_ar1(n, r, seed):
    if not isinstance(n, (int, np.integer)) or n < 2 or not abs(r) < 1:
        raise ValueError("Use an integer n >= 2 and abs(r) < 1.")
    rng = np.random.default_rng(seed)
    x = np.empty(n)
    x[0] = rng.normal()  # Draw from the stationary distribution.
    noise_scale = np.sqrt(1 - r**2)
    for t in range(1, n):
        x[t] = r * x[t - 1] + noise_scale * rng.normal()
    return x

N = 32_768
r = 0.9
x = stationary_ar1(N, r, seed=142)
iid = stationary_ar1(N, 0.0, seed=242)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(iid[:250], alpha=0.55, label="independent")
axes[0].plot(x[:250], label=f"correlated, r={r}")
axes[0].set(xlabel="Measurement index", ylabel="x")
axes[0].legend()
bins = np.linspace(-4, 4, 41)
axes[1].hist(iid, bins=bins, density=True, histtype="step", label="independent")
axes[1].hist(x, bins=bins, density=True, histtype="step", label="correlated")
axes[1].set(xlabel="x", ylabel="Probability density")
axes[1].legend()
plt.show()

The histograms can look nearly the same even when the time series look different. A histogram discards the order of the measurements.

**Predict:** which sequence should give a less precise estimate of the mean, for the same number of measurements?

## Autocorrelation: how long does the memory last?

At lag $k$, compare each measurement with the one $k$ steps later:

$$\rho(k)=\frac{\operatorname{Cov}(x_t,x_{t+k})}{\operatorname{Var}(x_t)}.$$

For this chain, $\rho(k)=r^k$. A value near 1 means strong positive correlation; a value near 0 means little linear correlation at that lag.

We estimate it from one run after subtracting the sample mean. The function uses the same normalization at every lag. Its FFT computes all the shifted products efficiently; zero padding prevents the end of the sequence from wrapping around to its beginning.

In [ ]:
def autocorrelation(values, max_lag):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or not np.all(np.isfinite(values)):
        raise ValueError("Supply a finite one-dimensional sequence.")
    if not isinstance(max_lag, (int, np.integer)) or not 0 <= max_lag < len(values):
        raise ValueError("max_lag must be an integer between 0 and N - 1.")
    centered = values - values.mean()
    if not np.any(centered):
        raise ValueError("Autocorrelation is undefined for a constant sequence.")
    fft_length = 1 << (2 * len(values) - 1).bit_length()
    transform = np.fft.rfft(centered, n=fft_length)
    products = np.fft.irfft(transform * transform.conj(), n=fft_length)
    return products[:max_lag + 1] / products[0]

lags = np.arange(121)
acf = autocorrelation(x, max_lag=120)
plt.plot(lags, acf, label="estimated from our run")
plt.plot(lags, r**lags, "--", label=r"known $r^k$")
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("Lag k (measurements)")
plt.ylabel("Autocorrelation")
plt.legend()
plt.show()

## Count information, not just measurements

For a long stationary run, define

$$g=1+2\sum_{k=1}^{\infty}\rho(k),\qquad
\tau_{\mathrm{int}}=\frac{g}{2},\qquad N_{\mathrm{eff}}\approx\frac{N}{g}.$$

Then $\operatorname{SE}(\bar x)\approx s\sqrt{g/N}$.
We use the convention $\tau_{\mathrm{int}}=\tfrac12+\sum_{k\ge1}\rho(k)$; some references instead call $g$ the autocorrelation time.

For AR(1), $g=(1+r)/(1-r)$. At $r=0.9$, $g=19$: 32,768 measurements contain roughly as much information about the mean as 1,725 independent ones.

The empirical autocorrelation becomes noisy at large lag. We sum only through a stated **window**; we do not add every noisy lag or treat a negative estimate of $g$ as meaningful.

In [ ]:
def exact_mean_se(n, r):
    # Finite-N result for this stationary, unit-variance chain.
    lag = np.arange(1, n)
    variance = (1 + 2 * np.sum((1 - lag / n) * r**lag)) / n
    return np.sqrt(variance)

window = 50
g_estimated = 1 + 2 * acf[1:window + 1].sum()
if g_estimated <= 0:
    raise ValueError("This ACF window does not give a usable error estimate.")
se_iid = x.std(ddof=1) / np.sqrt(N)
se_acf = se_iid * np.sqrt(g_estimated)
se_exact = exact_mean_se(N, r)
print(f"Mean: {x.mean():.4f} (known population mean: 0)")
print(f"SE assuming independence: {se_iid:.4f}")
print(f"SE from ACF, window {window}: {se_acf:.4f}")
print(f"Exact finite-N SE for this model: {se_exact:.4f}")
print(f"Estimated g: {g_estimated:.2f}; estimated N_eff: {N / g_estimated:.0f}")

The exact reference uses

$$\operatorname{Var}(\bar x)=\frac{1}{N}\left[1+2\sum_{k=1}^{N-1}
\left(1-\frac{k}{N}\right)r^k\right].$$

This is the variance of the mean across repeated runs. Our estimated error bars fluctuate because we only have one run. An estimate need not land within one standard error of the true value every time.

## Blocking: make larger units of data

Split the sequence into $K=N/B$ consecutive, nonoverlapping blocks of length $B$. Average within each block, then estimate

$$\operatorname{SE}_{B}(\bar x)=\frac{s(\bar x_1,\ldots,\bar x_K)}{\sqrt K}.$$

If blocks are much longer than the correlation scale, their means are approximately independent. Small blocks retain correlations; very large blocks leave too few independent units to estimate an error reliably.

We'll save both $\overline{x}_b$ and $\overline{x^2}_b$ for each block. The second column will let us analyze the variance later. Every observation is retained; block sizes must divide the data length exactly.

In [ ]:
def block_moments(values, block_size):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or not np.all(np.isfinite(values)):
        raise ValueError("Supply a finite one-dimensional sequence.")
    if (not isinstance(block_size, (int, np.integer)) or block_size < 1
            or len(values) % block_size or len(values) // block_size < 2):
        raise ValueError("Block size must divide N and leave at least two blocks.")
    blocks = values.reshape(-1, block_size)
    return np.column_stack((blocks.mean(axis=1), (blocks**2).mean(axis=1)))

block_sizes = 2 ** np.arange(0, 11)
blocked_errors = []
for size in block_sizes:
    means = block_moments(x, size)[:, 0]
    blocked_errors.append(means.std(ddof=1) / np.sqrt(len(means)))

plt.semilogx(block_sizes, blocked_errors, "o-", base=2, label="blocked estimate")
plt.axhline(se_exact, color="black", linestyle="--", label="exact mean SE")
plt.xlabel("Block length B (measurements)")
plt.ylabel("Estimated standard error of the mean")
plt.legend()
plt.show()

Look for a range where the error estimate becomes reasonably stable while many blocks remain. A noisy flat region is evidence to examine, not proof that all dependence is gone. A short or poorly mixed run may show no usable range at all.

**Exercise 1:** compare block lengths 1, 16, 256, and 1024. For each, report the number of blocks and the estimated error. Which estimates are near the exact reference? What do you lose by making blocks larger?

In [ ]:
for size in (1, 16, 256, 1024):
    means = block_moments(x, size)[:, 0]
    error = means.std(ddof=1) / np.sqrt(len(means))
    print(f"B={size:4d}, K={len(means):5d}, SE(mean)={error:.4f}")

## A nonlinear result: the variance

Now estimate the population variance:

$$\widehat V=\overline{x^2}-\bar x^2.$$

Its two terms come from the same data and fluctuate together. We need to preserve that connection when estimating its uncertainty.

The target here is $V=1$. This particular estimator has a small finite-sample bias: $\mathbb E[\widehat V]=1-\operatorname{Var}(\bar x)$. We'll report the direct estimate and its statistical error; we are not applying a bias correction.

Use the global averages of the two block columns. **Averaging the variances within blocks is a different calculation**: it loses fluctuations between block means.

In [ ]:
def variance_statistic(moments):
    return moments[..., 1] - moments[..., 0]**2

block_size = 256
moments = block_moments(x, block_size)
K = len(moments)
variance_estimate = variance_statistic(moments.mean(axis=0))
print(f"{K} blocks of {block_size} measurements")
print(f"Estimated variance: {variance_estimate:.4f}; population value: 1")
print(f"Known finite-N bias of this estimator: {-se_exact**2:.6f}")

## Jackknife: leave one block out

Remove block $b$, average the remaining rows, and recompute the **whole statistic**. This gives $K$ estimates $\widehat V_{(-b)}$.

With $\overline V_{(-)}$ their average, the jackknife standard error is

$$s_J=\sqrt{\frac{K-1}{K}\sum_{b=1}^{K}
\left(\widehat V_{(-b)}-\overline V_{(-)}\right)^2}.$$

We delete the same block from both moment columns. Leaving out individual correlated measurements would still underestimate the error. The block length must be adequate for the observable we are analyzing.

In [ ]:
def jackknife(block_stats, statistic):
    count = len(block_stats)
    if count < 2:
        raise ValueError("Jackknife requires at least two blocks.")
    leave_one_out = (block_stats.sum(axis=0) - block_stats) / (count - 1)
    replicates = statistic(leave_one_out)
    error = np.sqrt((count - 1) / count * np.sum((replicates - replicates.mean())**2))
    estimate = statistic(block_stats.mean(axis=0))
    return estimate, error, replicates

estimate, se_jackknife, jackknife_values = jackknife(moments, variance_statistic)
print(f"Variance = {estimate:.4f} +/- {se_jackknife:.4f} (block jackknife SE)")

## Bootstrap: sample blocks with replacement

Build another dataset by drawing $K$ block rows **with replacement**, then recompute $\widehat V$. Some blocks appear more than once; some are absent. Repeat and take the standard deviation of the resulting estimates.

Each row contains $[\overline{x}_b,\overline{x^2}_b]$. Resampling the row keeps the two moments paired and keeps the fluctuations already contained in each block. We treat different blocks as approximately independent.

More bootstrap replicates make the numerical resampling estimate steadier. They do **not** create more observations or cure a poorly mixed simulation.

In [ ]:
def bootstrap(block_stats, statistic, n_resamples, seed):
    if len(block_stats) < 2 or not isinstance(n_resamples, int) or n_resamples < 2:
        raise ValueError("Use at least two blocks and two resamples.")
    rng = np.random.default_rng(seed)
    count = len(block_stats)
    estimates = np.empty(n_resamples)
    for b in range(n_resamples):
        selected = rng.integers(0, count, size=count)
        estimates[b] = statistic(block_stats[selected].mean(axis=0))
    return estimates

bootstrap_values = bootstrap(moments, variance_statistic, n_resamples=2000, seed=314)
se_bootstrap = bootstrap_values.std(ddof=1)
print(f"Variance = {variance_estimate:.4f} +/- {se_bootstrap:.4f} (block bootstrap SE)")

## What goes wrong if we resample individual measurements?

For the default correlated chain (`r = 0.9`), the following comparison deliberately uses the wrong independence assumption. With block length 1, bootstrap sampling breaks the time correlations and gives a distribution of estimates that is too narrow. When you rerun with `r = 0`, the observations are independent and individual resampling becomes appropriate.

Both histograms are bootstrap distributions of the **variance estimate**, not histograms of the original measurements and not probabilities for the true variance.

In [ ]:
individual_moments = block_moments(x, 1)
iid_bootstrap_values = bootstrap(individual_moments, variance_statistic,
                                 n_resamples=400, seed=2718)
se_naive_variance = iid_bootstrap_values.std(ddof=1)
plt.hist(iid_bootstrap_values, bins=30, density=True, alpha=0.5,
         label="individual resampling (assumes independence)")
plt.hist(bootstrap_values, bins=40, density=True, alpha=0.5,
         label=f"block resampling, B={block_size}")
plt.axvline(variance_estimate, color="black", linestyle="--", label="sample estimate")
plt.xlabel("Resampled variance estimate")
plt.ylabel("Density across bootstrap replicates")
plt.legend(fontsize=10)
plt.show()

## Compare like with like

The errors for the **mean** and the errors for the **variance** answer different questions. An autocorrelation time belongs to an observable, not to the simulation as a whole. For this Gaussian model, $x_t^2$ has autocorrelation $r^{2k}$, different from $x_t$.

The table below keeps the two estimates separate. Jackknife and bootstrap should be broadly consistent when the blocks are suitable; they need not give identical numbers.

In [ ]:
se_block_mean = moments[:, 0].std(ddof=1) / np.sqrt(K)
print("STANDARD ERROR OF THE MEAN")
for name, error in [("assume independence", se_iid), ("ACF window", se_acf),
                    ("block means", se_block_mean), ("exact model reference", se_exact)]:
    print(f"  {name:26s} {error:.5f}")
print("\nSTANDARD ERROR OF THE VARIANCE ESTIMATE")
for name, error in [("individual bootstrap (iid assumption)", se_naive_variance),
                    ("block jackknife", se_jackknife), ("block bootstrap", se_bootstrap)]:
    print(f"  {name:29s} {error:.5f}")

## Exercise 2: change the dependence

Change `r` in the data-generation cell to **0.0**, then **0.5**, then back to **0.9**, and run the notebook again from that cell onward. Keep `N` fixed.

1. What changes in the trace, histogram, and autocorrelation plot?
2. Compare the standard deviation of the measurements with the standard error of their mean. Which changes most?
3. When does individual-measurement bootstrap become reasonable?

Then keep `r = 0.9` and try `block_size = 16`, `256`, and `1024` in the variance-analysis cell. Rerun from there through the comparison table. Report the estimate, its error, the block length, and the number of blocks. Explain your choice of block length.

## Back to the Ising model

For each equilibrated configuration, save the **signed magnetization per spin** $m=M/N_s$, where $N_s$ is the number of spins. Keep observations in their original sweep order. Our earlier $N$ counted measurements, not spins.

For field coupling $-hM$ and $k_B=1$, the magnetic susceptibility **per spin** is

$$\chi=\frac{N_s}{T}\left(\langle m^2\rangle-\langle m\rangle^2\right).$$

This has the same two-moment structure as our variance example. Block the rows $[\overline m_b,\overline{m^2}_b]$, then apply jackknife or bootstrap to the complete expression. Use the same block selections for both columns.

**Discuss:** why would replacing signed $m$ with $|m|$ change this quantity? Why can't an error estimate reveal a magnetic phase the simulation never visited?

## Optional: apply it to your Lab 11 measurements

In a copy of your Lab 11 notebook, use `block_moments`, `jackknife`, and `bootstrap` from above on the equilibrated signed magnetization trace. Divide total magnetization by the number of spins if needed.

Define the statistic as `N_spins / temperature * variance_statistic(moment_pair)` and pass it to the same resampling functions. Choose blocks after inspecting the autocorrelation and block-size dependence of the observables involved. If the data length is incompatible with your block size, make and document an explicit data-selection choice before calling `block_moments`.

Compare several block sizes. A small reported uncertainty from a trapped or unequilibrated chain is not evidence of a reliable physical result.

## Before reporting an error bar

- Check equilibration and exploration separately from uncertainty estimation.
- Inspect the ordered trace and the autocorrelation of the observable you need.
- Use blocks long enough to reduce dependence while retaining enough blocks to compare.
- Recompute the whole nonlinear statistic in each jackknife or bootstrap sample.
- Report the estimate, what the uncertainty means, the run length, and the block or window choice.

**Exit question:** you have 100,000 measurements but a long autocorrelation time. What would you check before claiming that your estimate is precise?

## References

- [Charles Geyer: MCMC and batch means](https://www.stat.umn.edu/geyer/5421/notes/mcmc-bayes.html).
- [Weigel and Janke: Error estimation and reduction with cross correlations](https://arxiv.org/abs/1002.4517), especially the jackknife treatment of correlated estimates.
- [Cosma Shalizi: Bootstrap](https://www.stat.cmu.edu/~cshalizi/dst/20/lectures/16/lecture-16.html).
- [Anders Sandvik: Monte Carlo methods](https://physics.bu.edu/~py502/lectures5/mc.pdf), for fluctuation observables in spin simulations.